# Exploration of BNF codes structure in prescriptions data

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
hl.init(sc=sc, default_reference='GRCh38')

#### Envinroment setup

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Input database configuration and loading

In [ ]:
db_name = 'clinical_phenos'
full_tb_name = 'full_phenos_hail_0.2.116.ht'

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database", project=dxpy.PROJECT_CONTEXT_ID)['id']
url = f"dnax://{db_uri}/{full_tb_name}"
full = hl.read_table(url)

### Checking dataset size

In [ ]:
full.count()

In [ ]:
%time bnf_df = full.filter(full.system == 'bnf').cache()
bnf_df.count()

### Adding neccessary helpers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

def perform_count_aggregation(grouped_by, input_data, aggregated = False):
    if aggregated:
        aggregated = grouped_by
    else:
        aggregated = grouped_by.aggregate(occurences=hl.agg.count())
    data_count = input_data.count()
    aggregated = aggregated.annotate(share = hl.format('%.3f%%', aggregated.occurences / hl.float(data_count) * 100))
    aggregated = aggregated.order_by(-aggregated.occurences).cache()
    return aggregated
    
def show_aggregated_examples(aggredated, source_data, column, sample_sz):
    source_data_cols = list(source_data.row.keys())
    source_data_cols.remove(column)
    source_data_cols.insert(0, column)
    aggregated_py = aggregated.collect()
    rand_seed = int(datetime.now().timestamp())
    source_data = source_data.annotate(rand = hl.rand_unif(0, 1, seed = rand_seed)).order_by('rand').cache()
    joined = source_data.head(0)
    for row in aggregated_py:
        joined = joined.union(source_data.filter(source_data[column] == row[column]).head(sample_sz))
    joined = joined.cache()
    joined = joined.key_by(column).join(aggredated.key_by(column), how = 'left')
    joined = joined.order_by(-joined.occurences).select(*source_data_cols).cache()
    joined.show(-1)

def aggregated_bar_plot(aggregated, column):
    aggregated_pd = aggregated.to_pandas()
    aggregated_pd['share_pct'] = aggregated_pd['share'].str.rstrip('%').astype(float)
    
    plt.figure(figsize=(8, 4))
    plt.bar(aggregated_pd[column], aggregated_pd['share_pct'], color = plt.cm.viridis(np.linspace(0, 1, 5)))
    plt.ylabel('Share %')

## Checking BNF codes length distribution

In [ ]:
%time bnf_df = bnf_df.annotate(code_len = hl.len(bnf_df.code)).cache()

In [ ]:
%time aggregated = perform_count_aggregation(bnf_df.group_by('code_len'), bnf_df)
aggregated.show(-1)

In [ ]:
# show_aggregated_examples(aggregated, bnf_df, 'code_len', 5)

In [ ]:
aggregated_bar_plot(aggregated.annotate(code_len = hl.str(aggregated.code_len)), 'code_len')
plt.title('BNF prescriptions code lenght')
plt.show()

## Checking BNF codes formats

BNF codes format in Biobank data falls into a few categories:

| Class name                | Format as regular expression                         | Example      | Description |
|---------------------------|---------------------------------------------|-----------------------|--|
| `TPP_dotted_subparagraph`   | `^\d{2}\.\d{2}\.\d{2}\.\d{2}\.00$`          | `04.08.01.03.00`      | Dotted code from TPP system for BNF Subparagraph. (most common) |
| `numeric_subparagraph`        | `^\d{8}$`                                   | `02050501`            | Not dotted BNF Subparagraph code. |
| `exact`                     | `^\d{7}[A-Z0-9]{8}$`                        | `0209000A0AAAJAJ`     | Excact (full) BNF code. |
| `dressings_appliance`       | `^\d{11}$`                                  | `21010230102`         | BNF code covering dressings and appliances. |
| `6_digit_group`             | `^\d{6}$`                                   | `040304`              | 6-digit group code. |
| `4_digit_group`             | `^\d{4}$`                                   | `0212`                | 4-digit group code. |
| `2_digit_group`             | `^\d{2}$`                                   | `23`                  | 6-digit group code. |
| `A_starting`                | `^A(\d{1}\|\d{3}\|\d{5}\|\d{7})$`              | `A2020201`            | Strange "A" letter starting codes. |

**Important notice:** Pay attention that for BNF Subparagraph codes (TPP dotted and 8-digid format) 7th digit is at 8th position and at 7th position there is always dummy "0".

There is one 1-digit code which falls into "others" class.


In [ ]:
code_formats = {
    'TPP_dotted_subparagraph': (r'^\d{2}\.\d{2}\.\d{2}\.\d{2}\.00$', '04.08.01.03.00'),
    'numeric_subparagraph': (r'^\d{8}$', '02050501'),
    'exact': (r'^\d{7}[A-Z0-9]{8}$', '0209000A0AAAJAJ'),
    'dressings_appliances': (r'^\d{11}$', '21010230102'),
    '6_digit_group': (r'^\d{6}$', '040304'),
    '4_digit_group': (r'^\d{4}$', '0212'),
    '2_digit_group': (r'^\d{2}$', '23'),
    'A_starting': (r'^A(\d{1}|\d{3}|\d{5}|\d{7})$', 'A2020201'),
}

In [ ]:
%%time
hl_code_formats = hl.literal([(code_formats[k][0], k) for k in code_formats.keys()])
bnf_df = bnf_df.annotate(
    code_format = hl.or_else(hl.find(lambda cformat: bnf_df.code.matches(cformat[0]), hl_code_formats), hl.literal(('.+', 'other')))[1]
).cache()

#### Examining unknown BNF format records

In [ ]:
other = bnf_df.filter(bnf_df.code_format == 'other').cache()
other.count()

In [ ]:
other.show()

#### BNF code formats prescriptions share

In [ ]:
%time aggregated = perform_count_aggregation(bnf_df.group_by('code_format'), bnf_df)
aggregated.show(-1)

In [ ]:
aggregated_bar_plot(aggregated, 'code_format')
plt.title('BNF prescriptions code formats')
plt.xticks(rotation=60)
plt.show()

#### Prescriptions examples of particular BNF code formats

In [ ]:
%time show_aggregated_examples(aggregated, bnf_df, 'code_format', 5)